# SC4001: Few-Shot Flower Recognition Evaluation

This notebook loads the best trained `ViT-VPT` checkpoint and evaluates it on the test set. It then generates the required `confusion_matrix` and `classification_report` for the final 10-page university report.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm

from models.vit_model import ViTForFlowerRecognition
from utils.data_loader import get_dataloaders

### 1. Initialize DataLoader and Model

We only need the `test_loader` for evaluation. The `batch_size` here only affects the speed of evaluation, not the results.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Get dataloaders (we only care about test_loader here)
_, _, test_loader = get_dataloaders(batch_size=32)

# Initialize model with the same architecture parameters as training
model = ViTForFlowerRecognition(num_classes=102, num_prompts=10)

# Load the best checkpoint
checkpoint_path = 'best_vit_vpt_model.pth'
try:
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Successfully loaded checkpoint from '{checkpoint_path}'")
except FileNotFoundError:
    print(f"Error: '{checkpoint_path}' not found. Make sure you have trained the model first.")

model = model.to(device)
model.eval()  # Set model to evaluation mode

### 2. Run Inference on Test Set

We iterate over the `test_loader`, accumulating all predictions and true labels to compute the final metrics.

In [ ]:
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc="Evaluating"):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        logits, _ = model(inputs)
        
        # Get predictions
        _, predicted = torch.max(logits.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

test_accuracy = np.mean(all_preds == all_targets)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

### 3. Generate Classification Report

The classification report shows precision, recall, and f1-score for each of the 102 flower classes.

In [ ]:
report = classification_report(all_targets, all_preds, zero_division=0)
print("Classification Report:")
print(report)

# Optional: Save to a text file for the university report
with open("classification_report.txt", "w") as f:
    f.write(report)
print("Report saved to classification_report.txt")

### 4. Plot Confusion Matrix

Since there are 102 classes, the confusion matrix will be quite large. We plot a high-resolution heatmap.

In [ ]:
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(24, 20))
sns.heatmap(cm, annot=False, cmap="Blues", cbar=True)
plt.title("Confusion Matrix (102 Classes)", fontsize=20)
plt.xlabel("Predicted Label", fontsize=16)
plt.ylabel("True Label", fontsize=16)

plt.tight_layout()
# Save the plot as a high-res PNG for the report
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()